# Predicting the Sale Price of Bulldozers using Machine Learning

In this notebook, we're going to go through an example machine learning project with the goal of predicting the sale price of bulldozers

## 1. Problem Definition

> How well can we predict the future sale price of a bulldozer, given its characteristics and previous examples of how much similar bulldozers have been sold for?

## 2. Data

The data is downloaded from the Kaggle Bluebook for Bulldozers competition:

There are 3 main datasets:

* Train.csv is the training set, which contains data through the end of 2011.

* Valid.csv is the validation set, which contains data from January 1, 2012 - April 30, 2012 You make predictions on this set throughout the majority of the competition. Your score on this set is used to create the public leaderboard.

* Test.csv is the test set, which won't be released until the last week of the competition. It contains data from May 1, 2012 - November 2012. Your score on the test set determines your final rank for the competition.

## 3. Evaluation

The evaluation metric for this competition is the RMSLE (root mean squared log error) between the actual and predicted auction prices.

For more evaluation of this project check: www.kaggle.com/competitions/bluebook-for-bulldozers/overview/evaluation

Note: The goal for most regression evaluation metrics is to minimize the error. For example, our goal for this project will be to build a machine laerning model which minimises RMSLE. 

## 4. Features

Kaggle provides a data dictionary detailing all of the features of the dataset.

In [ ]:
import os

# Just go one folder up to the project root
# Run this notebook from the repository root  

print("Now working in:", os.getcwd())
# Should show: C:\Users\DONKAMS\Desktop\Ml_projects\bulldozer-price-prediction-project

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

In [ ]:
# Import training and validation sets
import pandas as pd
df = pd.read_csv('data/TrainAndValid.csv',
                 low_memory=False)

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.columns

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df["saledate"][:1000], df["SalePrice"][:1000])

In [ ]:
df.saledate[:1000]

In [ ]:
df.SalePrice.plot.hist()

### Parsing dates

When we work with time series data, we want to enrich the time & date component as much as possible.

We can do that by telling pandas which of our columns has date in it using the `parse_dates` parameter.

In [ ]:
# Import data again but this time parse dates
df = pd.read_csv("data/TrainAndValid.csv",
                 low_memory=False,
                parse_dates=["saledate"])

In [ ]:
df.saledate.dtype

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df["saledate"][:1000], df["SalePrice"][:1000])

In [ ]:
df.head()

In [ ]:
df.head().T

In [ ]:
df.saledate.head(20)

### Sort DataFrame by saledate 

When working with time series data,it's a good idea to sort it by date



In [ ]:
# Sort Dataframe in date order
df.sort_values(by=["saledate"], inplace=True, ascending=True)
df.saledate.head(20)

### Make a copy of the original DataFrame

We make a copy of the original dataframeso when we manipulate the copy, we've stillgot our original data.

In [ ]:
# Make a copy
df_tmp = df.copy()

### Add datetime parameters for `saledate` column

In [ ]:
df_tmp["SaleYear"] = df_tmp.saledate.dt.year
df_tmp["SaleMonth"] = df_tmp.saledate.dt.month
df_tmp["SaleDay"] = df_tmp.saledate.dt.day
df_tmp["SaleDayOfWeek"] = df_tmp.saledate.dt.dayofweek
df_tmp["SaleDayOfYear"] = df_tmp.saledate.dt.dayofyear

In [ ]:
df_tmp.head().T

In [ ]:
# Now we've enriched our DatFrame with datetime features, we can remove `saledate`
df_tmp.drop("saledate", axis=1, inplace=True)

In [ ]:
# Check the values of different columns
df_tmp.state.value_counts()

In [ ]:
df_tmp.head()

## 5. Modelling

We've done enough EDA (we could always do more) but let's start to do some model-driven EDA

In [ ]:
# Let's build a machine learning model
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_jobs=-1,
                             random_state=42)
model.fit(df_tmp.drop("SalePrice", axis=1), df_tmp["SalePrice"])

In [ ]:
df_tmp.info()

In [ ]:
df_tmp.isna().sum()

### Convert string to categories

One way we can turn all of data into numbers is by converting them into pandas categories.

In [ ]:
df_tmp.head().T

In [ ]:
pd.api.types.is_string_dtype(df_tmp["UsageBand"])

In [ ]:
# Find the columns which contain strings
for label, content in df_tmp.items():
    if pd.api.types.is_string_dtype(content) or pd.api.types.is_object_dtype(content):
        print(label)

In [ ]:
# if you're wondering what df.items() does, here's an example
random_dict = {"key1": "hello",
              "key2": "world"}
for key, value in random_dict.items():
    print(f"this is a key: {key}",
          f"this is a value: {value}")

In [ ]:
# This will turn all of the string value into category values
for label, content in df_tmp.items():
    if pd.api.types.is_string_dtype(content) or pd.api.types.is_object_dtype(content):
        df_tmp[label] = content.astype("category").cat.as_ordered()

In [ ]:
df_tmp.info()

In [ ]:
df_tmp.state.cat.categories

In [ ]:
df_tmp.state.value_counts()

In [ ]:
df_tmp.state.cat.codes

Thanks to pandas Categories we now have a way to access all our data in the form of numbers.

But we still have a bunch of missing data...

In [ ]:
# Check missing data
df_tmp.isnull().sum()/len(df_tmp)

### Save preprocessed data

In [ ]:
# Export current tmp dataframe
df_tmp.to_csv("bulldozer-price-prediction-project/train_tmp.csv",
             index=False)

In [ ]:
# Import preprocessed data
df_tmp = pd.read_csv("bulldozer-price-prediction-project/train_tmp.csv",
                     low_memory=False)
df_tmp.head().T

## Fill missing values 

### Fill numerical missing values first

In [ ]:
# Checking for which are the numeric columns
for label, content in df_tmp.items():
    if pd.api.types.is_numeric_dtype(content):
        print(label)

In [ ]:
df_tmp.ModelID

In [ ]:
# Check for which numeric columns have null value
for label, content in df_tmp.items():
    if pd.api.types.is_numeric_dtype(content):
        if pd.isnull(content).sum():
            print(label)

In [ ]:
# Fill numeric rows with the median
for label, content in df_tmp.items():
    if pd.api.types.is_numeric_dtype(content):
        if pd.isnull(content).sum():
            # Add a binary column which tells us if the data was missing or not
            df_tmp[label+"_is_missing"] = pd.isnull(content)
            # Fill missing numeric values with median
            df_tmp[label] = content.fillna(content.median())

In [ ]:
# Demonstrate how median is more robust than mean
hundreds = np.full((1000,), 100)
hundreds_billion = np.append(hundreds, 1000000000)
np.mean(hundreds), np.mean(hundreds_billion), np.median(hundreds), np.median(hundreds_billion)

In [ ]:
# Check if there's any null numeric values
for label, content in df_tmp.items():
    if pd.api.types.is_numeric_dtype(content):
        if pd.isnull(content).sum():
            print(label)

In [ ]:
# Check to see how many examples we're missing
df_tmp.auctioneerID_is_missing.value_counts()

In [ ]:
df_tmp.isna().sum()

### Filling and turning categorical variables into numbers

In [ ]:
# Check for columns which aren't numeric
for label, content in df_tmp.items():
    if not pd.api.types.is_numeric_dtype(content):
        print(label)

In [ ]:
# Turn categorical variables into numbers and fill missing
for label, content in df_tmp.items():
    if not pd.api.types.is_numeric_dtype(content):
        # Add binary column to indicate whether sample had missing value
        df_tmp[label+"_is_missing"] = pd.isnull(content)
        # Turn categories into numbers and add +1
        df_tmp[label] = pd.Categorical(content).codes+1

In [ ]:
pd.Categorical(df_tmp["state"]).codes+1

In [ ]:
df_tmp.info()

In [ ]:
df_tmp.head().T

In [ ]:
df_tmp.isna().sum()

Now that all of our data is numeric as well as our dataframe has no missing value, we should be able to build a machine learning model.

In [ ]:
df_tmp.head()

In [ ]:
len(df_tmp)

In [ ]:
%%time
# Instantiate model 
model = RandomForestRegressor(n_jobs=-1,
                              random_state=42)

# Fit the model
model.fit(df_tmp.drop("SalePrice", axis=1), df_tmp["SalePrice"])

In [ ]:
# Score the model
model.score(df_tmp.drop("SalePrice", axis=1), df_tmp["SalePrice"])

**Question:** Why doesn't the above metric hold water? (why isn't the metric reliable)

### Splitting data into train/validation sets

In [ ]:
df_tmp.SaleYear

In [ ]:
df_tmp.SaleYear.value_counts()

In [ ]:
# Split data into training and validation
df_val = df_tmp[df_tmp.SaleYear == 2012]
df_train = df_tmp[df_tmp.SaleYear != 2012]

len(df_val), len(df_train)

In [ ]:
# Split data into X & y
X_train, y_train = df_train.drop("SalePrice", axis=1), df_train.SalePrice
X_valid, y_valid = df_val.drop("SalePrice", axis=1), df_val.SalePrice

X_train.shape, y_train.shape, X_valid.shape, y_valid.shape

In [ ]:
y_train

### Building an evaluation function

In [ ]:
# Create evaluation function (the competition uses RMSLE)
from sklearn.metrics import mean_squared_log_error, mean_absolute_error, r2_score

def rmsle(y_test, y_preds):
    """
    Calculates root mean squared log error between predictions and true lables.
    """
    return np.sqrt(mean_squared_log_error(y_test, y_preds))

# Create function to evaluate model on a few different levels
def show_scores(model):
    train_preds = model.predict(X_train)
    val_preds = model.predict(X_valid)
    scores = {"Training MAE": mean_absolute_error(y_train, train_preds),
             "Valid MAE": mean_absolute_error(y_valid, val_preds),
             "Training RMSLE": rmsle(y_train, train_preds),
             "Valid RMSLE": rmsle(y_valid, val_preds),
             "Training R^2": r2_score(y_train, train_preds),
             "Valid R^2": r2_score(y_valid, val_preds)}
    return scores

### Testing our model on a subset (to tune the hyperparameters)

In [ ]:
# This takes far too long..for experimenting

# %%time
# model = RandomForestRegeressor(n_jobs=-1,
#                               random_state=42)

# model.fit(X_train, y_train)

In [ ]:
len(X_train)

In [ ]:
# Change max_samples value
model = RandomForestRegressor(n_jobs=-1,
                             random_state=42,
                             max_samples=10000)
model

In [ ]:
%%time
# Cutting down on the max number of samples each estimator can see improves training time
model.fit(X_train, y_train)

In [ ]:
X_train.shape[0] * 100

In [ ]:
show_scores(model)

### Hyparameter tuning with RandomizedSearchCV

In [ ]:
%%time
from sklearn.model_selection import RandomizedSearchCV

# Different RandomForesRegressor hyperparameters
rf_grid = {"n_estimators": np.arange(10, 100, 10),
           "max_depth": [None, 3, 5, 10],
           "min_samples_split": np.arange(2, 20, 20),
           "min_samples_leaf": np.arange(1, 20, 2),
           "max_features": [0.5, 1, "sqrt"],
           "max_samples":[10000]}

# Instantiate RandomizedSearchCV model
rs_model = RandomizedSearchCV(RandomForestRegressor(n_jobs=-1,
                                                    random_state=42),
                             param_distributions=rf_grid,
                             n_iter=10,
                             cv=5,
                             verbose=True)
#Fit the RandomizedSearchCV model
rs_model.fit(X_train, y_train)

In [ ]:
# Find the best model hyperparameters
rs_model.best_params_

In [ ]:
# Evaluate the RandomizedSearch model
show_scores(rs_model)

### Train a model with the best hyperparameters

**Note:** These were found after 100 iterations of `RandomizedSearchCV`

In [ ]:
%%time

# Most ideal hyperparameters
ideal_model = RandomForestRegressor(n_estimators=40,
                                    min_samples_leaf=1,
                                    min_samples_split=14,
                                    max_features=0.5,
                                    n_jobs=-1,
                                    max_samples=None,
                                    random_state=42) # random state so our results are reproducable

# Fit the ideal model
ideal_model.fit(X_train, y_train)

In [ ]:
# Scores for ideal model (trained on all the data)
show_scores(ideal_model)

In [ ]:
# Scores on rs_model (only trained on -10,000 examples)
show_scores(rs_model)

## Make predictions on test data

In [ ]:
# Import the test data
df_test = pd.read_csv(r"C:\Users\DONKAMS\Desktop\Ml_projects\bulldozer-price-prediction-project\data\bluebook-for-bulldozers\Test.csv",
                      low_memory=False,
                      parse_dates=["saledate"])
df_test.head()

In [ ]:
# Make predictions on the test dataset
test_preds = ideal_model.predict(df_test)

### Preprocessing the data (getting the test dataset in the same format as our training dataset)

In [ ]:
def preprocess_data(df):
    # Date features (must match exactly what you did on training)
    df["SaleYear"] = df["saledate"].dt.year
    df["SaleMonth"] = df["saledate"].dt.month
    df["SaleDay"] = df["saledate"].dt.day
    df["SaleDayOfWeek"] = df["saledate"].dt.dayofweek
    df["SaleDayOfYear"] = df["saledate"].dt.dayofyear
    
    df.drop("saledate", axis=1, inplace=True)
    
    # Handle ALL columns exactly like training 
    for label, content in df.items():
        # Numeric columns
        if pd.api.types.is_numeric_dtype(content):
            if content.isna().sum():
                df[label + "_is_missing"] = content.isna()
                df[label] = content.fillna(content.median())
        
        # Categorical / object columns
        else:
            df[label + "_is_missing"] = content.isna()
            df[label] = pd.Categorical(content).codes + 1
    
    return df

In [ ]:
# Reload test data fresh
df_test = pd.read_csv(r"C:\Users\DONKAMS\Desktop\Ml_projects\bulldozer-price-prediction-project\data\bluebook-for-bulldozers\Test.csv",
                      low_memory=False,
                      parse_dates=["saledate"])

# Apply the CORRECT preprocessing function
df_test = preprocess_data(df_test)
df_test.head()

In [ ]:
X_train.head()

In [ ]:
# Make predictions on updated test data
test_preds = ideal_model.predict(df_test)

In [ ]:
# We can find how the columns differ using sets
set(X_train.columns) - set(df_test.columns)

In [ ]:
# Manually adjust df_test to have auctioneer_ID is missing
df_test["auctioneerID_is_missing"] = False
df_test.head()

Finally our test dataframe has the same features as our training dataframe, we can make predictions!

In [ ]:
df_test = df_test[X_train.columns]
df_test.head()

In [ ]:
# Make predictions on the test data
test_preds = ideal_model.predict(df_test)

In [ ]:
len(test_preds)

In [ ]:
# Format predictions into the same format Kaggle is after
df_preds = pd.DataFrame()
df_preds["SalesID"] = df_test["SalesID"]
df_preds["SalesPrice"] = test_preds
df_preds

In [ ]:
# Export prediction data
df_preds.to_csv(r"C:\Users\DONKAMS\Desktop\Ml_projects\data/test_predictions.csv", index=False)

### Feature Importance

Feature importance seeks to figure out which different attributes of the data were most important when it comes to predicting **target variable** (SalePrice).  

In [ ]:
# Find feature importance of our best model
ideal_model.feature_importances_

In [ ]:
# Helper function for plotting feature importance 
def plot_features(columns, importances, n=20):
    df = (pd.DataFrame({"features": columns,
                       "feature_importances": importances})
         .sort_values("feature_importances", ascending=False)
         .reset_index(drop=True))

    # Plot the dataframe
    fig, ax = plt.subplots()
    ax.barh(df["features"][:n], df["feature_importances"][:20])
    ax.set_ylabel("Features")
    ax.set_xlabel("Feature importance")
    ax.invert_yaxis()

In [ ]:
plot_features(X_train.columns, ideal_model.feature_importances_)

In [ ]:
df["Enclosure"].value_counts()

**Question to finish:** Why might knowing the feature importance of a trained machine learning model be helpful

**Final challenge:** What other machine learning models could you try on our dataset? Hint: https://scikit-learn.org/stable/machine_learning_map.html check out the regression section of this map, or try to look at something like CatBoost.ai or XGBoost.ai.